# NewsBot — ejecución completa en Google Colab (sin subir ZIP)
**Student:** José Miguel Tuozzo Izaguirre | **Course:** ITAI 2373 — NLP

Este notebook descarga la versión pública más reciente de [NewsBot](https://github.com/jtuozzo1616/Mid-Term-Jose-Group-NewsBot-Intelligence-System), instala recursos lingüísticos, procesa el **BBC News Train.csv** etiquetado y genera evidencias reales en PDF/CSV/PNG. No depende del ZIP anterior.

En Colab selecciona **Runtime → Run all**. Si falla la copia automática del dataset, te pedirá cargar **BBC News Train.csv** descargado de [Kaggle](https://www.kaggle.com/competitions/learn-ai-bbc/data).

**Cuidado:** no se inventan métricas; la generación se detiene ante errores. La reflexión sigue siendo un borrador que debe revisarse y personalizarse.

In [ ]:
from pathlib import Path
import os, sys, subprocess
URL="https://github.com/jtuozzo1616/Mid-Term-Jose-Group-NewsBot-Intelligence-System.git"
PROJECT=Path("/content/NewsBot_Project")
if PROJECT.exists() and (PROJECT/".git").is_dir():
    subprocess.run(["git","-C",str(PROJECT),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1",URL,str(PROJECT)],check=True)
os.chdir(PROJECT)
assert (PROJECT/"run_all_and_package.py").is_file()
print("✓ Proyecto descargado desde:",URL)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(PROJECT/"requirements.txt")],check=True)
import spacy, nltk
try:
    spacy.load("en_core_web_sm")
except OSError:
    subprocess.run([sys.executable,"-m","spacy","download","en_core_web_sm"],check=True)
assert nltk.download("vader_lexicon",quiet=True)
print("✓ Dependencias y modelos NLP disponibles")

In [ ]:
from pathlib import Path
import urllib.request, pandas as pd
TARGET=PROJECT/"data"/"BBC News Train.csv"
TARGET.parent.mkdir(exist_ok=True,parents=True)
if not TARGET.is_file():
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/alvinrach/learn-ai-bbc/main/BBC%20News%20Train.csv",TARGET)
    except Exception:
        print("Copia pública no disponible. Descarga BBC News Train.csv de Kaggle y cárgalo aquí.")
        from google.colab import files
        upload=files.upload()
        source=next((n for n in upload if n.lower().strip()=="bbc news train.csv"),None)
        if source is None: raise FileNotFoundError("Se necesita BBC News Train.csv con etiquetas.")
        TARGET.write_bytes(upload[source])
data=pd.read_csv(TARGET)
assert {"ArticleId","Category"}.issubset(data.columns) and ({"Text","Article"} & set(data.columns))
assert 500 <= len(data) <= 2000
print("✓ BBC etiquetado:",len(data),"artículos;",data["Category"].nunique(),"categorías")
del data

In [ ]:
import subprocess, sys, os
RUN=subprocess.run([sys.executable,"-u","run_all_and_package.py",
                    "--github","https://github.com/jtuozzo1616/Mid-Term-Jose-Group-NewsBot-Intelligence-System","--timeout","1800"],cwd=str(PROJECT))
if RUN.returncode: raise RuntimeError("La ejecución falló; lee el error de esta celda. No hay resultados verificados.")
print("✓ Notebook completo, informes y evidencias terminados")

In [ ]:
import json
from IPython.display import Image, display
from verify_release import verify
manifest=verify(PROJECT/"outputs")
print("Artículos:",manifest["rows"])
print("Modelo:",manifest["selected_model"])
print("Macro-F1 de test:",round(manifest["holdout_macro_f1"],4))
print("Accuracy de test:",round(manifest["holdout_accuracy"],4))
for n in ("05_model_cv_comparison.png","06_confusion_matrix.png","08_robustness_stress_test.png"):
    display(Image(filename=str(PROJECT/"outputs"/n),width=850))

In [ ]:
from pathlib import Path
from google.colab import files
PACKAGE=PROJECT/"NewsBot_VERIFIED_EXECUTION_EVIDENCE.zip"
assert PACKAGE.exists() and PACKAGE.stat().st_size>0
print("Descargando evidencia:",PACKAGE.name)
files.download(str(PACKAGE))